<a href="https://colab.research.google.com/github/DanielYaari28/AIPI510P1/blob/daniel-eda/notebooks/college_scorecard_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import pandas as pd

file_path = "/content/Most-Recent-Cohorts-Institution.csv"

df = pd.read_csv(file_path, low_memory=False)

print(df.shape)
df.head()

(6273, 3308)


,UNITID,OPEID,OPEID6,INSTNM,CITY,STABBR,ZIP,ACCREDAGENCY,INSTURL,NPCURL,...,MD_EARN_WNE_INC1_P11,MD_EARN_WNE_INC2_P11,MD_EARN_WNE_INC3_P11,MD_EARN_WNE_INDEP0_P11,MD_EARN_WNE_INDEP1_P11,MD_EARN_WNE_MALE0_P11,MD_EARN_WNE_MALE1_P11,SCORECARD_SECTOR,EARN_THR_STATE,EARN_THR_NAT
0,100654,100200.0,1002.0,Alabama A & M University,Normal,AL,35762,Southern Association of Colleges and Schools C...,www.aamu.edu/,www.aamu.edu/admissions-aid/tuition-fees/net-p...,...,36650.0,41070.0,47016.0,38892.0,41738.0,38167.0,40250.0,4,32204.0,36082.0
1,100663,105200.0,1052.0,University of Alabama at Birmingham,Birmingham,AL,35294-0110,Southern Association of Colleges and Schools C...,https://www.uab.edu/,https://tcc.ruffalonl.com/University of Alabam...,...,47182.0,51896.0,54368.0,50488.0,51505.0,46559.0,59181.0,4,32204.0,36082.0
2,100690,2503400.0,25034.0,Amridge University,Montgomery,AL,36117-3553,Southern Association of Colleges and Schools C...,https://www.amridgeuniversity.edu/,https://www2.amridgeuniversity.edu:9091/,...,35752.0,41007.0,NaN,NaN,38467.0,32654.0,49435.0,5,32204.0,36082.0
3,100706,105500.0,1055.0,University of Alabama in Huntsville,Huntsville,AL,35899,Southern Association of Colleges and Schools C...,www.uah.edu/,uah.clearcostcalculator.com/student/default/ne...,...,51208.0,62219.0,62577.0,55920.0,60221.0,47787.0,67454.0,4,32204.0,36082.0
4,100724,100500.0,1005.0,Alabama State University,Montgomery,AL,36104-0271,Southern Association of Colleges and Schools C...,www.alasu.edu/,tcc.ruffalonl.com/Alabama State University/Fre...,...,32844.0,36932.0,37966.0,34294.0,31797.0,32303.0,36964.0,4,32204.0,36082.0


In [13]:
#Institutions considerations:
eligible = df[
    (df["PREDDEG"] == 3) & #undergrad
    (df["UGDS"] >= 2000) #above 2,000 undergrads
].copy()

print("Eligible universities:", len(eligible))

Eligible universities: 828


In [14]:
project_cols = [
    "UNITID",
    "INSTNM",
    "STABBR",
    "CONTROL",
    "UGDS",

    # Student Success
    "C150_4",
    "RET_FT4",

    # Affordability
    "NPT4_PUB",
    "NPT4_PRIV",
    "DEBT_MDN",
    "COSTT4_A",

    # Outcomes
    "MD_EARN_WNE_P10",

    # Accessibility
    "PCTPELL",

    # Context
    "ADM_RATE"
]

project_df = eligible[project_cols].copy()

In [15]:
project_df.info()

print("\nMissing values:")
print(project_df.isna().sum())

<class 'pandas.core.frame.DataFrame'>
Index: 828 entries, 0 to 5640
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   UNITID           828 non-null    int64  
 1   INSTNM           828 non-null    object 
 2   STABBR           828 non-null    object 
 3   CONTROL          828 non-null    int64  
 4   UGDS             828 non-null    float64
 5   C150_4           820 non-null    float64
 6   RET_FT4          820 non-null    float64
 7   NPT4_PUB         460 non-null    float64
 8   NPT4_PRIV        355 non-null    float64
 9   DEBT_MDN         823 non-null    object 
 10  COSTT4_A         811 non-null    float64
 11  MD_EARN_WNE_P10  820 non-null    float64
 12  PCTPELL          825 non-null    float64
 13  ADM_RATE         759 non-null    float64
dtypes: float64(9), int64(2), object(3)
memory usage: 97.0+ KB

Missing values:
UNITID               0
INSTNM               0
STABBR               0
CONTROL       

In [16]:
project_df["NET_PRICE"] = (
    project_df["NPT4_PUB"]
    .fillna(project_df["NPT4_PRIV"])
)

In [17]:
print("Missing net price:", project_df["NET_PRICE"].isna().sum())

print(
    "Net price coverage:",
    round(project_df["NET_PRICE"].notna().mean() * 100, 1),
    "%"
)

Missing net price: 13
Net price coverage: 98.4 %


In [18]:
numeric_cols = [
    "C150_4",
    "RET_FT4",
    "NET_PRICE",
    "DEBT_MDN",
    "COSTT4_A",
    "MD_EARN_WNE_P10",
    "PCTPELL",
    "ADM_RATE"
]

project_df[numeric_cols].describe()

,C150_4,RET_FT4,NET_PRICE,COSTT4_A,MD_EARN_WNE_P10,PCTPELL,ADM_RATE
count,820.000000,820.000000,815.000000,811.000000,820.000000,825.000000,759.000000
mean,0.598426,0.787361,20614.119018,40120.606658,59968.714634,0.332919,0.705403
std,0.187398,0.118789,10109.545139,21879.910768,16314.657289,0.162240,0.240027
min,0.000000,0.000000,2984.000000,11837.000000,24328.000000,0.000000,0.036100
25%,0.473300,0.721875,13524.500000,23858.500000,49288.250000,0.210900,0.600250
50%,0.593950,0.791750,18059.000000,29483.000000,57296.500000,0.304600,0.776400
75%,0.735800,0.869100,26376.500000,55626.000000,68092.750000,0.411000,0.881900
max,0.976100,1.000000,58741.000000,93512.000000,143372.000000,0.992800,0.999200


In [19]:
project_df[project_df["DEBT_MDN"].isna()][["INSTNM", "DEBT_MDN"]]

,INSTNM,DEBT_MDN
516,United States Air Force Academy,NaN
1321,United States Naval Academy,NaN
2161,United States Military Academy,NaN
2642,Grove City College,NaN
5257,University of the People,NaN


In [20]:
project_df[project_df["DEBT_MDN"].isna()][["INSTNM",
"C150_4",
"RET_FT4",
"MD_EARN_WNE_P10",
"PCTPELL"]]

,INSTNM,C150_4,RET_FT4,MD_EARN_WNE_P10,PCTPELL
516,United States Air Force Academy,0.8751,0.9655,NaN,NaN
1321,United States Naval Academy,0.9294,0.9732,NaN,NaN
2161,United States Military Academy,0.8706,0.9463,NaN,NaN
2642,Grove City College,0.8317,0.8833,NaN,0.0
5257,University of the People,0.3354,NaN,NaN,0.0


In [21]:
ranking_vars = [
    "C150_4",
    "RET_FT4",
    "NET_PRICE",
    "DEBT_MDN",
    "MD_EARN_WNE_P10",
    "PCTPELL"
]

In [22]:
missing_data = project_df[ranking_vars].isna().any(axis=1)
project_df[missing_data][["INSTNM"] + ranking_vars]

,INSTNM,C150_4,RET_FT4,NET_PRICE,DEBT_MDN,MD_EARN_WNE_P10,PCTPELL
7,Athens State University,NaN,NaN,NaN,14861,50273.0,0.4251
454,Walden University,NaN,0.0000,33817.0,11500,42810.0,0.5518
516,United States Air Force Academy,0.8751,0.9655,NaN,NaN,NaN,NaN
1321,United States Naval Academy,0.9294,0.9732,NaN,NaN,NaN,NaN
1820,Beth Medrash Govoha,NaN,NaN,NaN,5500,47544.0,0.7449
1886,Thomas Edison State University,NaN,NaN,NaN,7813,69331.0,0.2367
1981,CUNY Graduate School and University Center,NaN,NaN,NaN,9800,65991.0,0.3846
2155,Excelsior University,NaN,NaN,NaN,10870,78493.0,0.2824
2161,United States Military Academy,0.8706,0.9463,NaN,NaN,NaN,NaN
2642,Grove City College,0.8317,0.8833,NaN,NaN,NaN,0.0000


In [23]:
complete_df = project_df[~missing_data].copy()

print(len(complete_df))

809
